<a href="https://colab.research.google.com/github/anmolpatilcs24-ux/AI-LAB/blob/main/IDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import heapq
import copy

class PuzzleNode:
    def __init__(self, board, parent=None, action=None, g=0, h=0):
        self.board = board       # 2D tuple (3x3) representing the board
        self.parent = parent
        self.action = action     # Action taken to reach this state
        self.g = g               # Path cost
        self.h = h               # Heuristic cost
        self.f = g + h           # Total cost

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

def get_manhattan_distance(board, goal):
    distance = 0
    goal_positions = {}
    for r in range(3):
        for c in range(3):
            goal_positions[goal[r][c]] = (r, c)

    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0:
                target_r, target_c = goal_positions[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def get_neighbors(node, goal):
    neighbors = []
    r, c = node.get_blank_pos()
    moves = {
        'UP': (r - 1, c),
        'DOWN': (r + 1, c),
        'LEFT': (r, c - 1),
        'RIGHT': (r, c + 1)
    }

    for action, (nr, nc) in moves.items():
        if 0 <= nr < 3 and 0 <= nc < 3:
            # Create new board layout
            new_board = [list(row) for row in node.board]
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
            new_board_tuple = tuple(tuple(row) for row in new_board)

            h = get_manhattan_distance(new_board_tuple, goal)
            neighbors.append(PuzzleNode(new_board_tuple, node, action, node.g + 1, h))
    return neighbors

def solve_8_puzzle(start_board, goal_board):
    start_node = PuzzleNode(start_board, g=0, h=get_manhattan_distance(start_board, goal_board))

    open_list = []
    heapq.heappush(open_list, start_node)
    closed_set = set()

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.board == goal_board:
            path = []
            curr = current_node
            while curr.parent is not None:
                path.append((curr.action, curr.board))
                curr = curr.parent
            path.reverse()
            return path

        closed_set.add(current_node.board)

        for neighbor in get_neighbors(current_node, goal_board):
            if neighbor.board in closed_set:
                continue
            heapq.heappush(open_list, neighbor)

    return None

In [9]:
# Ensure the function is defined and then solve
try:
    custom_solution = solve_8_puzzle(custom_start, custom_goal)
except NameError:
    # If not defined, we fallback or remind to run the definition cell
    print("Please run the code cell containing the 'solve_8_puzzle' function definition first, then run this cell again.")
    custom_solution = None

if custom_solution:
    print(f"Solved in {len(custom_solution)} steps!\n")
    print("Initial State:")
    for row in custom_start:
        print(row)
    print("\nSteps to reach goal:\n")
    for step, (action, board) in enumerate(custom_solution, 1):
        print(f"Step {step}: Move blank space {action}")
        for row in board:
            print(row)
        print()

Solved in 6 steps!

Initial State:
(2, 8, 3)
(1, 6, 4)
(0, 7, 5)

Steps to reach goal:

Step 1: Move blank space RIGHT
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)

Step 2: Move blank space UP
(2, 8, 3)
(1, 0, 4)
(7, 6, 5)

Step 3: Move blank space UP
(2, 0, 3)
(1, 8, 4)
(7, 6, 5)

Step 4: Move blank space LEFT
(0, 2, 3)
(1, 8, 4)
(7, 6, 5)

Step 5: Move blank space DOWN
(1, 2, 3)
(0, 8, 4)
(7, 6, 5)

Step 6: Move blank space RIGHT
(1, 2, 3)
(8, 0, 4)
(7, 6, 5)



In [10]:
def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j


def get_neighbors(state):

    neighbors = []

    blank_row, blank_col = find_blank(state)


    directions = [
        (0, 1),
        (1, 0),
        (0, -1),
        (-1, 0)
    ]

    for dr, dc in directions:

        new_row = blank_row + dr
        new_col = blank_col + dc

        if 0 <= new_row < 3 and 0 <= new_col < 3:

            new_state = [row[:] for row in state]

            new_state[blank_row][blank_col], \
            new_state[new_row][new_col] = \
            new_state[new_row][new_col], \
            new_state[blank_row][blank_col]

            neighbors.append(new_state)

    return neighbors



def state_to_tuple(state):
    return tuple(tuple(row) for row in state)




def is_solvable(state):

    values = []

    for row in state:
        for value in row:
            if value != 0:
                values.append(value)

    inversions = 0

    for i in range(len(values)):
        for j in range(i + 1, len(values)):

            if values[i] > values[j]:
                inversions += 1

    return inversions % 2 == 0




def dfs(initial_state, goal_state):

    initial = state_to_tuple(initial_state)
    goal = state_to_tuple(goal_state)


    stack = [initial]


    visited = {initial}


    parent = {initial: None}

    while stack:

        current = stack.pop()


        if current == goal:
            return reconstruct_path(parent, goal)

        current_list = [list(row) for row in current]

        neighbors = get_neighbors(current_list)

        for next_state in neighbors:

            next_tuple = state_to_tuple(next_state)

            if next_tuple not in visited:

                visited.add(next_tuple)


                parent[next_tuple] = current

                stack.append(next_tuple)

    return None




def reconstruct_path(parent, goal):

    path = []

    current = goal

    while current is not None:

        path.append(current)

        current = parent[current]


    path.reverse()

    return path




def depth_limited_dfs(state, goal, limit, path):


    if state == goal:
        return path


    if limit == 0:
        return None


    state_list = [list(row) for row in state]

    neighbors = get_neighbors(state_list)


    path_set = set(path)

    for next_state in neighbors:

        next_tuple = state_to_tuple(next_state)

        if next_tuple not in path_set:

            result = depth_limited_dfs(
                next_tuple,
                goal,
                limit - 1,
                path + [next_tuple]
            )

            if result is not None:
                return result

    return None



def ids(initial_state, goal_state):

    initial = state_to_tuple(initial_state)
    goal = state_to_tuple(goal_state)

    depth_limit = 0

    while True:

        print("Searching with depth limit =", depth_limit)

        result = depth_limited_dfs(
            initial,
            goal,
            depth_limit,
            [initial]
        )

        if result is not None:
            return result

        depth_limit += 1




def display_solution(path):

    print("\n========== SOLUTION ==========\n")

    for step, state in enumerate(path):

        print("Step", step, ":")

        for row in state:
            print(list(row))

        print()


    path_cost = len(path) - 1

    print("Path Cost:", path_cost)
    print("Number of States:", len(path))
    print("Name: Anmol")
    print("USN : 1WN24CS044")






print("\nEnter the initial state.")
print("Use 0 for the blank space.\n")

initial_state = []

for i in range(3):

    row = list(
        map(
            int,
            input("Row " + str(i + 1) + ": ").split()
        )
    )

    initial_state.append(row)




print("\nEnter the goal state.")
print("Use 0 for the blank space.\n")

goal_state = []

for i in range(3):

    row = list(
        map(
            int,
            input("Row " + str(i + 1) + ": ").split()
        )
    )

    goal_state.append(row)




if not is_solvable(initial_state):

    print("\nThe given puzzle is not solvable.")

else:


    print("\nChoose the algorithm:")
    print("1. Depth First Search (DFS)")
    print("2. Iterative Deepening Search (IDS)")

    choice = int(input("\nEnter your choice: "))




    if choice == 1:

        print("\nSolving using DFS...\n")

        solution = dfs(
            initial_state,
            goal_state
        )

        if solution is not None:

            display_solution(solution)
            print("\nTime Complexity for DFS: O(b^d) where b is branching factor, d is depth of solution.")
            print("Space Complexity for DFS: O(bd) in worst case.")

        else:

            print("No solution found using DFS.")
            print("\nTime Complexity for DFS: O(b^d) where b is branching factor, d is depth of solution.")
            print("Space Complexity for DFS: O(bd) in worst case.")



    elif choice == 2:

        print("\nSolving using IDS\n")

        solution = ids(
            initial_state,
            goal_state
        )

        if solution is not None:

            display_solution(solution)
            print("\nTime Complexity for IDS: O(b^d) where b is branching factor, d is depth of solution.")
            print("Space Complexity for IDS: O(bd) as it re-explores states, but only stores path for current DLS iteration.")

        else:

            print("No solution found using IDS.")
            print("\nTime Complexity for IDS: O(b^d) where b is branching factor, d is depth of solution.")
            print("Space Complexity for IDS: O(bd) as it re-explores states, but only stores path for current DLS iteration.")


    else:

        print("Invalid choice.")


Enter the initial state.
Use 0 for the blank space.

Row 1: 2 8 3
Row 2: 1 6 4
Row 3: 0 7 5

Enter the goal state.
Use 0 for the blank space.

Row 1: 1 2 3 
Row 2: 4 5 6
Row 3: 7 8 0

The given puzzle is not solvable.


In [11]:
print("Anmol, 1WN24CS044")
def get_blank_pos(board):
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c

def get_neighbors(board):
    neighbors = []
    r, c = get_blank_pos(board)
    moves = {
        'UP': (r - 1, c),
        'DOWN': (r + 1, c),
        'LEFT': (r, c - 1),
        'RIGHT': (r, c + 1)
    }

    for action, (nr, nc) in moves.items():
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_board = [list(row) for row in board]
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]
            neighbors.append((action, tuple(tuple(row) for row in new_board)))
    return neighbors

def depth_limited_search(board, goal, limit, path, visited):
    if board == goal:
        return path
    if limit <= 0:
        return None

    visited.add(board)
    for action, neighbor in get_neighbors(board):
        if neighbor not in visited:
            new_path = path + [(action, neighbor)]
            result = depth_limited_search(neighbor, goal, limit - 1, new_path, visited)
            if result is not None:
                return result
    visited.remove(board)
    return None

def ids_8_puzzle(start, goal, max_depth=50):
    for depth in range(max_depth):
        visited = set()
        result = depth_limited_search(start, goal, depth, [], visited)
        if result is not None:
            return result, depth
    return None, max_depth

# Test the IDS solver
ids_start = (
    (1, 2, 3),
    (0, 4, 6),
    (7, 5, 8)
)

ids_goal = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

ids_solution, solved_depth = ids_8_puzzle(ids_start, ids_goal)

if ids_solution:
    print(f"Solved using IDS at Depth Limit: {solved_depth} ({len(ids_solution)} steps!)\n")
    print("Initial State:")
    for row in ids_start:
        print(row)
    print("\nSteps to reach goal:\n")
    for step, (action, board) in enumerate(ids_solution, 1):
        print(f"Step {step}: Move blank space {action}")
        for row in board:
            print(row)
        print()
else:
    print("No solution found within depth limit.")

Anmol, 1WN24CS044
Solved using IDS at Depth Limit: 3 (3 steps!)

Initial State:
(1, 2, 3)
(0, 4, 6)
(7, 5, 8)

Steps to reach goal:

Step 1: Move blank space RIGHT
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 2: Move blank space DOWN
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Step 3: Move blank space RIGHT
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

